#  **DimUser**

###  Autoloader

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import  *

In [0]:
df_user = spark.readStream.format("cloudFiles")\
            .option("cloudFiles.format","parquet")\
            .option("cloudFiles.schemaLocation","abfss://silver@storageaccountspotifyy.dfs.core.windows.net/DimUser/schema")\
            .option("schemaEvolutionMode","addNewColumns")\
            .load("abfss://bronze@storageaccountspotifyy.dfs.core.windows.net/DimUser")

In [0]:
df_user = df_user.dropDuplicates(['user_id'])

In [0]:
df_user.writeStream.format("delta")\
            .outputMode("append")\
            .option("checkpointLocation","abfss://silver@storageaccountspotifyy.dfs.core.windows.net/DimUser/chekpoint")\
            .trigger(once=True)\
            .option("mergeSchema", True)\
            .option("path","abfss://silver@storageaccountspotifyy.dfs.core.windows.net/DimUser/data")\
            .toTable("azure_project_spotify.silver.DimUser")

In [0]:
%sql
SELECT * FROM azure_project_spotify.silver.dimuser

# **DimArtist**

In [0]:
df_artists = spark.readStream.format("cloudFiles")\
    .option("cloudFiles.format", "parquet")\
        .option("cloudFiles.schemaLocation", "abfss://silver@storageaccountspotifyy.dfs.core.windows.net/DimArtist/schema")\
        .load("abfss://bronze@storageaccountspotifyy.dfs.core.windows.net/DimArtist/")


In [0]:
df_artists = df_artists.dropDuplicates(['artist_id'])

In [0]:
df_artists.writeStream.format("delta")\
    .outputMode("append")\
    .option("checkpointLocation", "abfss://silver@storageaccountspotifyy.dfs.core.windows.net/DimArtist/chekpoint")\
    .trigger(once=True)\
    .option("path", "abfss://silver@storageaccountspotifyy.dfs.core.windows.net/DimArtist/data")\
    .toTable("azure_project_spotify.silver.DimArtist")

# **DimTrack**

In [0]:
df_track = spark.readStream.format("cloudFiles")\
    .option("cloudFiles.format", "parquet")\
    .option("cloudFiles.schemaLocation", "abfss://silver@storageaccountspotifyy.dfs.core.windows.net/DimTrack/schema")\
    .option("schemaEvolutionMode", "addNewColumns")\
    .load("abfss://bronze@storageaccountspotifyy.dfs.core.windows.net/DimTrack/")


In [0]:
df_track = df_track.withColumn("flag", when(col("duration_sec")<150, "low")\
                                    .when(col("duration_sec")>150, "medium")\
                                    .when(col("duration_sec")>200, "high")\
                                    .otherwise("other"))
df_track = df_track.withColumn("track_name",regexp_replace(col('track_name'),'-',' '))

In [0]:
df_track.writeStream.format("delta")\
    .outputMode("append")\
    .option("checkpointLocation", "abfss://silver@storageaccountspotifyy.dfs.core.windows.net/DimTrack/chekpoint")\
    .trigger(once=True)\
    .option("path", "abfss://silver@storageaccountspotifyy.dfs.core.windows.net/DimTrack/data")\
    .toTable("azure_project_spotify.silver.DimTrack")

In [0]:
%sql
SELECT * FROM azure_project_spotify.silver.dimtrack;

# **DimDate**

In [0]:
df_date = spark.readStream.format("cloudFiles")\
    .option("cloudFiles.format", "parquet")\
    .option("cloudFiles.schemaLocation", "abfss://silver@storageaccountspotifyy.dfs.core.windows.net/DimDate/schema")\
    .option("schemaEvolutionMode", "addNewColumn")\
    .load("abfss://bronze@storageaccountspotifyy.dfs.core.windows.net/DimDate/")


In [0]:
df_date.writeStream.format("delta")\
    .outputMode("append")\
    .option("checkpointLocation", "abfss://silver@storageaccountspotifyy.dfs.core.windows.net/DimDate/checkpoint")\
    .trigger(once=True)\
    .option("path", "abfss://silver@storageaccountspotifyy.dfs.core.windows.net/DimDate/data")\
    .toTable("azure_project_spotify.silver.DimDate")

In [0]:
%sql
SELECT * FROM azure_project_spotify.silver.dimdate

# **FactStream**

In [0]:
df_stream = spark.readStream.format("cloudFiles")\
    .option("cloudFiles.format", "parquet")\
    .option("cloudFiles.schemaLocation", "abfss://silver@storageaccountspotifyy.dfs.core.windows.net/FactStream/schema")\
    .option("schemaEvolutionMode", "addNewColumns")\
    .load("abfss://bronze@storageaccountspotifyy.dfs.core.windows.net/FactStream/")


In [0]:
df_stream.writeStream.format("delta")\
    .outputMode("append")\
    .option("checkpointLocation", "abfss://silver@storageaccountspotifyy.dfs.core.windows.net/FactStream/checkpoint")\
    .trigger(once=True)\
    .option("path", "abfss://silver@storageaccountspotifyy.dfs.core.windows.net/FactStream/data")\
    .toTable("azure_project_spotify.silver.FactStream")